<a href="https://colab.research.google.com/github/MrDev333/PhishingDetect/blob/backend/notebooks/ResolveEmail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Đọc tệp CSV "/content/Phishing_Email.csv" và trích xuất các đặc trưng từ nội dung email.

## Tải dữ liệu

### Subtask:
Tải dữ liệu từ tệp CSV được cung cấp vào DataFrame.


**Reasoning**:
The first step is to load the data from the CSV file into a pandas DataFrame.



In [1]:
import pandas as pd

df = pd.read_csv("/content/Phishing_Email.csv")

**Reasoning**:
Display the first few rows of the DataFrame to verify the data has been loaded correctly.



In [2]:
display(df.head())

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


## Khám phá dữ liệu

### Subtask:
Hiển thị thông tin cơ bản về dữ liệu, bao gồm các cột và một vài hàng đầu tiên.


**Reasoning**:
Display the first 5 rows of the DataFrame and the DataFrame's information to understand its structure and data types.



In [3]:
display(df.head())
display(df.info())

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18650 entries, 0 to 18649
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  18650 non-null  int64 
 1   Email Text  18634 non-null  object
 2   Email Type  18650 non-null  object
dtypes: int64(1), object(2)
memory usage: 437.2+ KB


None

## Trích xuất đặc trưng

### Subtask:
Áp dụng các kỹ thuật xử lý ngôn ngữ tự nhiên để trích xuất các đặc trưng có liên quan từ cột nội dung email.


**Reasoning**:
Handle missing values in the 'Email Text' column by filling them with an empty string and then apply TfidfVectorizer to transform the text data into a TF-IDF feature matrix.



In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['Email Text'] = df['Email Text'].fillna('')

tfidf_vectorizer = TfidfVectorizer()

tfidf_features = tfidf_vectorizer.fit_transform(df['Email Text'])

## Summary:

### Data Analysis Key Findings

*   The dataset contains 18650 rows and 3 columns: "Unnamed: 0", "Email Text", and "Email Type".
*   The column "Email Text" has missing values, with only 18634 non-null entries out of 18650.
*   The missing values in the "Email Text" column were successfully handled by filling them with empty strings.
*   TF-IDF features were successfully extracted from the "Email Text" column using `TfidfVectorizer`.

### Insights or Next Steps

*   The extracted TF-IDF features are ready to be used for training a machine learning model to classify emails as phishing or not.


# Task
Đọc dữ liệu từ tệp CSV "/content/Phishing_Email.csv", trích xuất các đặc trưng TF-IDF từ nội dung email, trích xuất các URL bằng thư viện `urlextract` và xử lý chúng theo logic trong notebook "/content/ResolveURLmk3.ipynb", sau đó kết hợp các đặc trưng đã trích xuất.

## Trích xuất và xử lý url

### Subtask:
Sử dụng thư viện `urlextract` để trích xuất URL từ nội dung email và áp dụng logic xử lý URL từ notebook `ResolveURLmk3.ipynb`.


**Reasoning**:
Import the `URLExtract` class and initialize an object to extract URLs from the 'Email Text' column. Then, recreate the URL processing logic from the specified notebook to generate new features based on the extracted URLs.



In [5]:
from urlextract import URLExtract
from urllib.parse import urlparse

extractor = URLExtract()

def extract_and_process_urls(text):
    urls = extractor.find_urls(text)
    url_features = {
        'num_urls': len(urls),
        'avg_url_length': 0,
        'avg_dots_in_url': 0,
        'avg_special_chars_in_url': 0,
        'has_ip_address_in_url': 0,
        'has_shortening_service': 0
    }

    if urls:
        total_length = 0
        total_dots = 0
        total_special_chars = 0
        special_chars = set("!@#$%^&*()_-+=[]{};:'\",.<>/?\\|`~")

        for url in urls:
            total_length += len(url)
            total_dots += url.count('.')
            total_special_chars += sum(1 for char in url if char in special_chars)

            parsed_url = urlparse(url)
            if parsed_url.hostname:
                # Check for IP address in hostname
                try:
                    parts = parsed_url.hostname.split('.')
                    if len(parts) == 4 and all(0 <= int(part) <= 255 for part in parts):
                        url_features['has_ip_address_in_url'] = 1
                except ValueError:
                    pass # Not an IP address

                # Check for common URL shortening services (this is a basic check)
                shortening_services = ['bit.ly', 'goo.gl', 'tinyurl.com', 'ow.ly', 't.co']
                if any(service in parsed_url.hostname for service in shortening_services):
                    url_features['has_shortening_service'] = 1


        url_features['avg_url_length'] = total_length / len(urls)
        url_features['avg_dots_in_url'] = total_dots / len(urls)
        url_features['avg_special_chars_in_url'] = total_special_chars / len(urls)

    return pd.Series(url_features)

url_features_df = df['Email Text'].apply(extract_and_process_urls)

display(url_features_df.head())

ModuleNotFoundError: No module named 'urlextract'

**Reasoning**:
The previous command failed because the `urlextract` library is not installed. Install the missing library using pip.



In [6]:
%pip install urlextract

**Reasoning**:
Now that the `urlextract` library is installed, re-run the code to extract and process URLs from the 'Email Text' column.



In [7]:
from urlextract import URLExtract
from urllib.parse import urlparse
import pandas as pd

extractor = URLExtract()

def extract_and_process_urls(text):
    urls = extractor.find_urls(text)
    url_features = {
        'num_urls': len(urls),
        'avg_url_length': 0,
        'avg_dots_in_url': 0,
        'avg_special_chars_in_url': 0,
        'has_ip_address_in_url': 0,
        'has_shortening_service': 0
    }

    if urls:
        total_length = 0
        total_dots = 0
        total_special_chars = 0
        special_chars = set("!@#$%^&*()_-+=[]{};:'\",.<>/?\\|`~")

        for url in urls:
            total_length += len(url)
            total_dots += url.count('.')
            total_special_chars += sum(1 for char in url if char in special_chars)

            parsed_url = urlparse(url)
            if parsed_url.hostname:
                # Check for IP address in hostname
                try:
                    parts = parsed_url.hostname.split('.')
                    if len(parts) == 4 and all(0 <= int(part) <= 255 for part in parts):
                        url_features['has_ip_address_in_url'] = 1
                except ValueError:
                    pass # Not an IP address

                # Check for common URL shortening services (this is a basic check)
                shortening_services = ['bit.ly', 'goo.gl', 'tinyurl.com', 'ow.ly', 't.co']
                if any(service in parsed_url.hostname for service in shortening_services):
                    url_features['has_shortening_service'] = 1


        url_features['avg_url_length'] = total_length / len(urls)
        url_features['avg_dots_in_url'] = total_dots / len(urls)
        url_features['avg_special_chars_in_url'] = total_special_chars / len(urls)

    return pd.Series(url_features)

url_features_df = df['Email Text'].apply(extract_and_process_urls)

display(url_features_df.head())

,num_urls,avg_url_length,avg_dots_in_url,avg_special_chars_in_url,has_ip_address_in_url,has_shortening_service
0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0
3,2.0,19.0,1.5,4.5,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0


## Kết hợp đặc trưng

### Subtask:
Kết hợp các đặc trưng TF-IDF và các đặc trưng liên quan đến URL.


**Reasoning**:
Import the hstack function from scipy.sparse and convert the url_features_df DataFrame into a NumPy array. Then, use hstack to combine the sparse matrix tfidf_features with the NumPy array of URL features to create a new sparse matrix containing all features.



In [8]:
from scipy.sparse import hstack

url_features_array = url_features_df.to_numpy()

combined_features = hstack([tfidf_features, url_features_array])

## Summary:

### Data Analysis Key Findings

*   The process successfully extracted and processed URLs from the email text using the `urlextract` library, creating features such as the number of URLs, average URL length, average dots, and flags for IP addresses or shortening services.
*   The TF-IDF features, which were presumably generated in a prior step not shown, were successfully combined with the extracted URL features.
*   The combination resulted in a single sparse matrix named `combined_features`, ready for subsequent analysis or modeling.

### Insights or Next Steps

*   The `combined_features` matrix is now ready to be used as input for training a machine learning model to classify phishing emails.
*   Further analysis could involve evaluating the importance of the newly added URL features in predicting phishing emails.


# Task
Đọc dữ liệu từ tệp CSV "/content/Phishing_Email.csv", trích xuất các đặc trưng TF-IDF từ nội dung email, trích xuất và xử lý URL từ nội dung email bằng cách sử dụng logic từ notebook "/content/ResolveURLmk3.ipynb", kết hợp các đặc trưng này và chọn các đặc trưng liên quan nhất để xác định email đáng ngờ.

## Tải dữ liệu

### Subtask:
Tải dữ liệu từ tệp CSV được cung cấp vào DataFrame.


**Reasoning**:
The first step is to load the data from the CSV file into a pandas DataFrame.



In [9]:
df = pd.read_csv("/content/Phishing_Email.csv")

## Khám phá dữ liệu

### Subtask:
Hiển thị thông tin cơ bản về dữ liệu, bao gồm các cột và một vài hàng đầu tiên.


**Reasoning**:
Display the first 5 rows of the DataFrame and the DataFrame's information to understand its structure and data types.



In [10]:
display(df.head())
display(df.info())

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18650 entries, 0 to 18649
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  18650 non-null  int64 
 1   Email Text  18634 non-null  object
 2   Email Type  18650 non-null  object
dtypes: int64(1), object(2)
memory usage: 437.2+ KB


None

## Trích xuất đặc trưng tf-idf

### Subtask:
Áp dụng TfidfVectorizer để trích xuất các đặc trưng TF-IDF từ cột nội dung email.


**Reasoning**:
Import the TfidfVectorizer class, handle missing values in the 'Email Text' column by filling them with an empty string, initialize a TfidfVectorizer object, and transform the 'Email Text' column into TF-IDF features.



In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['Email Text'] = df['Email Text'].fillna('')

tfidf_vectorizer = TfidfVectorizer()

tfidf_features = tfidf_vectorizer.fit_transform(df['Email Text'])

## Trích xuất và xử lý url

### Subtask:
Sử dụng thư viện `urlextract` để trích xuất URL từ nội dung email và áp dụng logic xử lý URL từ notebook `ResolveURLmk3.ipynb`.


**Reasoning**:
Re-run the code to extract and process URLs from the 'Email Text' column now that `urlextract` is installed.



In [13]:
from urlextract import URLExtract
from urllib.parse import urlparse
import pandas as pd

extractor = URLExtract()

def extract_and_process_urls(text):
    urls = extractor.find_urls(text)
    url_features = {
        'num_urls': len(urls),
        'avg_url_length': 0,
        'avg_dots_in_url': 0,
        'avg_special_chars_in_url': 0,
        'has_ip_address_in_url': 0,
        'has_shortening_service': 0
    }

    if urls:
        total_length = 0
        total_dots = 0
        total_special_chars = 0
        special_chars = set("!@#$%^&*()_-+=[]{};:'\",.<>/?\\|`~")

        for url in urls:
            total_length += len(url)
            total_dots += url.count('.')
            total_special_chars += sum(1 for char in url if char in special_chars)

            parsed_url = urlparse(url)
            if parsed_url.hostname:
                # Check for IP address in hostname
                try:
                    parts = parsed_url.hostname.split('.')
                    if len(parts) == 4 and all(0 <= int(part) <= 255 for part in parts):
                        url_features['has_ip_address_in_url'] = 1
                except ValueError:
                    pass # Not an IP address

                # Check for common URL shortening services (this is a basic check)
                shortening_services = ['bit.ly', 'goo.gl', 'tinyurl.com', 'ow.ly', 't.co']
                if any(service in parsed_url.hostname for service in shortening_services):
                    url_features['has_shortening_service'] = 1


        url_features['avg_url_length'] = total_length / len(urls)
        url_features['avg_dots_in_url'] = total_dots / len(urls)
        url_features['avg_special_chars_in_url'] = total_special_chars / len(urls)

    return pd.Series(url_features)

url_features_df = df['Email Text'].apply(extract_and_process_urls)

display(url_features_df.head())

,num_urls,avg_url_length,avg_dots_in_url,avg_special_chars_in_url,has_ip_address_in_url,has_shortening_service
0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0
3,2.0,19.0,1.5,4.5,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0


## Kết hợp đặc trưng

### Subtask:
Kết hợp các đặc trưng TF-IDF và các đặc trưng liên quan đến URL.

**Reasoning**:
Import the hstack function from scipy.sparse and convert the url_features_df DataFrame into a NumPy array. Then, use hstack to combine the sparse matrix tfidf_features with the NumPy array of URL features to create a new sparse matrix containing all features.

In [14]:
from scipy.sparse import hstack

url_features_array = url_features_df.to_numpy()

combined_features = hstack([tfidf_features, url_features_array])

# Task
Đọc dữ liệu từ tệp CSV "/content/Phishing_Email.csv", trích xuất các đặc trưng TF-IDF từ nội dung email, trích xuất và xử lý URL từ nội dung email bằng cách sử dụng thư viện `urlextract` và logic từ notebook "/content/ResolveURLmk3.ipynb", kết hợp các đặc trưng này và chọn các đặc trưng có liên quan nhất để xác định email đáng ngờ.

## Tải dữ liệu

### Subtask:
Tải dữ liệu từ tệp CSV được cung cấp vào DataFrame.


**Reasoning**:
Load the data from the CSV file into a pandas DataFrame.



In [15]:
df = pd.read_csv("/content/Phishing_Email.csv")

## Lựa chọn đặc trưng

### Subtask:
Chọn các đặc trưng có liên quan nhất để xác định email đáng ngờ.


**Reasoning**:
Prepare the target variable by mapping the 'Email Type' column to numerical values, initialize a SelectKBest object with the chi2 score function to select the top 1000 features, apply SelectKBest to the combined features and the target variable, and display the shape of the selected features.



In [16]:
from sklearn.feature_selection import SelectKBest, chi2

y = df['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})

selector = SelectKBest(score_func=chi2, k=1000)

selected_features = selector.fit_transform(combined_features, y)

display(selected_features.shape)

(18650, 1000)

## Summary:

### Data Analysis Key Findings

*   The dataset containing phishing email information was successfully loaded from a CSV file.
*   TF-IDF features were extracted from the email content.
*   URLs were extracted and processed from the email content using the `urlextract` library and custom logic.
*   The extracted TF-IDF features and processed URL features were combined into a single feature set.
*   Feature selection using `SelectKBest` with the chi-squared test was applied to the combined feature set, resulting in the selection of the top 1000 features from the original set.
*   The target variable 'Email Type' was successfully mapped to numerical values (0 for 'Safe Email' and 1 for 'Phishing Email').

### Insights or Next Steps

*   The selected 1000 features are now ready to be used for training a machine learning model to classify emails as safe or phishing.
*   Further analysis could involve evaluating different machine learning algorithms on this selected feature set to determine the most effective model for phishing email detection.


# Task
Phân tích tệp CSV "/content/Phishing_Email.csv" để trích xuất các đặc trưng từ nội dung email, bao gồm cả việc trích xuất và xử lý URL bằng cách sử dụng logic từ notebook "/content/ResolveURLmk3.ipynb", sau đó chọn các đặc trưng liên quan nhất để xác định các email đáng ngờ và suy ra các quy tắc nhận dạng email đáng ngờ dựa trên các đặc trưng đã chọn.

## Tải dữ liệu

### Subtask:
Tải dữ liệu từ tệp CSV được cung cấp vào DataFrame.


**Reasoning**:
Load the data from the CSV file into a pandas DataFrame.



In [17]:
df = pd.read_csv("/content/Phishing_Email.csv")

## Huấn luyện mô hình đơn giản

### Subtask:
Huấn luyện một mô hình phân loại đơn giản (ví dụ: Logistic Regression) trên các đặc trưng đã chọn và nhãn email.


**Reasoning**:
Import LogisticRegression, create an instance, and fit the model using the selected features and the target variable.



In [18]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(selected_features, y)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

## Phân tích đặc trưng và xác định quy tắc

### Subtask:
Phân tích các trọng số (coefficients) hoặc tầm quan trọng của đặc trưng từ mô hình đã huấn luyện để xác định các đặc trưng nào đóng góp nhiều nhất vào việc phân loại và suy ra các quy tắc nhận dạng email đáng ngờ.


**Reasoning**:
Get the coefficients from the trained Logistic Regression model, get the feature names from the TF-IDF vectorizer and the URL features, create a DataFrame to store features and their coefficients, sort the DataFrame by the absolute values of the coefficients, and display the top and bottom features to identify the most important ones.



In [21]:
import numpy as np

coefficients = model.coef_[0]

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
url_feature_names = url_features_df.columns.tolist()
all_feature_names = list(tfidf_feature_names) + url_feature_names

selected_indices = selector.get_support(indices=True)
selected_feature_names = [all_feature_names[i] for i in selected_indices]

feature_importance = pd.DataFrame({'feature': selected_feature_names, 'coefficient': coefficients})

feature_importance['abs_coefficient'] = np.abs(feature_importance['coefficient'])

sorted_feature_importance = feature_importance.sort_values(by='abs_coefficient', ascending=False)

display("Top 10 most important features (positive coefficients):")
display(sorted_feature_importance[sorted_feature_importance['coefficient'] > 0].head(10))

display("Top 10 most important features (negative coefficients):")
display(sorted_feature_importance[sorted_feature_importance['coefficient'] < 0].head(10))

'Top 10 most important features (positive coefficients):'

,feature,coefficient,abs_coefficient
989,your,9.194365,9.194365
623,our,5.713258,5.713258
201,company,5.686334,5.686334
394,here,5.611319,5.611319
988,you,5.343440,5.343440
838,statements,5.120842,5.120842
191,click,5.094713,5.094713
569,money,5.008314,5.008314
352,free,4.892743,4.892743
998,has_ip_address_in_url,4.476312,4.476312


'Top 10 most important features (negative coefficients):'

,feature,coefficient,abs_coefficient
305,enron,-11.976672,11.976672
285,ect,-7.228782,7.228782
601,on,-5.311412,5.311412
879,thanks,-5.019949,5.019949
939,vince,-4.925430,4.925430
28,2001,-4.749605,4.749605
475,language,-4.656219,4.656219
916,university,-4.522769,4.522769
881,the,-4.367349,4.367349
27,2000,-4.298495,4.298495


**Reasoning**:
The error indicates that the list of feature names and the list of coefficients have different lengths. This is likely because the feature selection step reduced the number of features, but the original TF-IDF and URL feature names are still being used. We need to get the feature names corresponding to the selected features. We can achieve this by identifying which features were selected using the `get_support()` method of the selector and then filtering the original feature names. Then, create a DataFrame to store features and their coefficients, sort the DataFrame by the absolute values of the coefficients, and display the top and bottom features to identify the most important ones.



In [20]:
selected_indices = selector.get_support(indices=True)
selected_feature_names = [all_feature_names[i] for i in selected_indices]

feature_importance = pd.DataFrame({'feature': selected_feature_names, 'coefficient': coefficients})

feature_importance['abs_coefficient'] = np.abs(feature_importance['coefficient'])

sorted_feature_importance = feature_importance.sort_values(by='abs_coefficient', ascending=False)

display("Top 10 most important features (positive coefficients):")
display(sorted_feature_importance[sorted_feature_importance['coefficient'] > 0].head(10))

display("Top 10 most important features (negative coefficients):")
display(sorted_feature_importance[sorted_feature_importance['coefficient'] < 0].head(10))

'Top 10 most important features (positive coefficients):'

,feature,coefficient,abs_coefficient
989,your,9.194365,9.194365
623,our,5.713258,5.713258
201,company,5.686334,5.686334
394,here,5.611319,5.611319
988,you,5.343440,5.343440
838,statements,5.120842,5.120842
191,click,5.094713,5.094713
569,money,5.008314,5.008314
352,free,4.892743,4.892743
998,has_ip_address_in_url,4.476312,4.476312


'Top 10 most important features (negative coefficients):'

,feature,coefficient,abs_coefficient
305,enron,-11.976672,11.976672
285,ect,-7.228782,7.228782
601,on,-5.311412,5.311412
879,thanks,-5.019949,5.019949
939,vince,-4.925430,4.925430
28,2001,-4.749605,4.749605
475,language,-4.656219,4.656219
916,university,-4.522769,4.522769
881,the,-4.367349,4.367349
27,2000,-4.298495,4.298495


## Summary:

### Data Analysis Key Findings

*   The data was successfully loaded from the "Phishing\_Email.csv" file into a pandas DataFrame.
*   A Logistic Regression model was trained on selected features and the email labels, though a `ConvergenceWarning` was observed, suggesting potential issues with convergence or data scaling.
*   Analysis of the model's coefficients revealed that features such as 'your', 'our', 'company', 'here', 'you', 'statements', 'click', 'money', 'free', and 'has\_ip\_address\_in\_url' are strongly associated with phishing emails (positive coefficients).
*   Features like 'enron', 'ect', 'on', 'thanks', 'vince', and year numbers (e.g., '2001') are strongly associated with legitimate emails within this dataset (negative coefficients).

### Insights or Next Steps

*   The presence of certain words related to urgency, personal/company information requests, and specific URL characteristics are strong indicators for identifying potentially suspicious emails.
*   To improve model performance and address the `ConvergenceWarning`, consider scaling the features and/or increasing the `max_iter` parameter for the Logistic Regression model. Further steps should include evaluating the model's performance using appropriate metrics and refining the inferred rules based on domain expertise.


# Task
Phân tích tệp CSV "/content/Phishing_Email.csv" để trích xuất các đặc trưng từ nội dung email, bao gồm cả việc trích xuất và xử lý URL bằng cách sử dụng logic từ notebook "/content/ResolveURLmk3.ipynb". Sau đó, chọn các đặc trưng có liên quan nhất để xác định các email đáng ngờ và xác định các quy tắc dựa trên các đặc trưng này.

## Tải dữ liệu

### Subtask:
Tải dữ liệu từ tệp CSV được cung cấp vào DataFrame.


**Reasoning**:
The first step is to load the data from the CSV file into a pandas DataFrame.



In [22]:
df = pd.read_csv("/content/Phishing_Email.csv")

## Khám phá dữ liệu

### Subtask:
Hiển thị thông tin cơ bản về dữ liệu, bao gồm các cột và một vài hàng đầu tiên.


**Reasoning**:
Display the first 5 rows of the DataFrame and the DataFrame's information to understand its structure and data types.



In [23]:
display(df.head())
display(df.info())

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18650 entries, 0 to 18649
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  18650 non-null  int64 
 1   Email Text  18634 non-null  object
 2   Email Type  18650 non-null  object
dtypes: int64(1), object(2)
memory usage: 437.2+ KB


None

## Trích xuất đặc trưng tf-idf

### Subtask:
Áp dụng TfidfVectorizer để trích xuất các đặc trưng TF-IDF từ cột nội dung email.


**Reasoning**:
Import the TfidfVectorizer class, handle missing values in the 'Email Text' column by filling them with an empty string, initialize a TfidfVectorizer object, and transform the 'Email Text' column into TF-IDF features.



In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['Email Text'] = df['Email Text'].fillna('')

tfidf_vectorizer = TfidfVectorizer()

tfidf_features = tfidf_vectorizer.fit_transform(df['Email Text'])

## Trích xuất và xử lý url

### Subtask:
Sử dụng thư viện `urlextract` để trích xuất URL từ nội dung email và áp dụng logic xử lý URL từ notebook `ResolveURLmk3.ipynb`.


**Reasoning**:
Import the URLExtract class and urlparse function, initialize URLExtract, define the function to extract and process URLs based on the provided logic, apply the function to the 'Email Text' column to create a new DataFrame of URL features, and display the head of the new DataFrame.



In [25]:
from urlextract import URLExtract
from urllib.parse import urlparse
import pandas as pd

extractor = URLExtract()

def extract_and_process_urls(text):
    urls = extractor.find_urls(text)
    url_features = {
        'num_urls': len(urls),
        'avg_url_length': 0,
        'avg_dots_in_url': 0,
        'avg_special_chars_in_url': 0,
        'has_ip_address_in_url': 0,
        'has_shortening_service': 0
    }

    if urls:
        total_length = 0
        total_dots = 0
        total_special_chars = 0
        special_chars = set("!@#$%^&*()_-+=[]{};:'\",.<>/?\\|`~")

        for url in urls:
            total_length += len(url)
            total_dots += url.count('.')
            total_special_chars += sum(1 for char in url if char in special_chars)

            parsed_url = urlparse(url)
            if parsed_url.hostname:
                # Check for IP address in hostname
                try:
                    parts = parsed_url.hostname.split('.')
                    if len(parts) == 4 and all(0 <= int(part) <= 255 for part in parts):
                        url_features['has_ip_address_in_url'] = 1
                except ValueError:
                    pass # Not an IP address

                # Check for common URL shortening services (this is a basic check)
                shortening_services = ['bit.ly', 'goo.gl', 'tinyurl.com', 'ow.ly', 't.co']
                if any(service in parsed_url.hostname for service in shortening_services):
                    url_features['has_shortening_service'] = 1


        url_features['avg_url_length'] = total_length / len(urls)
        url_features['avg_dots_in_url'] = total_dots / len(urls)
        url_features['avg_special_chars_in_url'] = total_special_chars / len(urls)

    return pd.Series(url_features)

url_features_df = df['Email Text'].apply(extract_and_process_urls)

display(url_features_df.head())

,num_urls,avg_url_length,avg_dots_in_url,avg_special_chars_in_url,has_ip_address_in_url,has_shortening_service
0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0
3,2.0,19.0,1.5,4.5,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0


## Kết hợp đặc trưng

### Subtask:
Kết hợp các đặc trưng TF-IDF và các đặc trưng liên quan đến URL.


**Reasoning**:
Import the hstack function from scipy.sparse and convert the url_features_df DataFrame into a NumPy array. Then, use hstack to combine the sparse matrix tfidf_features with the NumPy array of URL features to create a new sparse matrix containing all features.



In [26]:
from scipy.sparse import hstack

url_features_array = url_features_df.to_numpy()

combined_features = hstack([tfidf_features, url_features_array])

## Lựa chọn đặc trưng

### Subtask:
Chọn các đặc trưng có liên quan nhất để xác định email đáng ngờ.


**Reasoning**:
Prepare the target variable by mapping the 'Email Type' column to numerical values, initialize a SelectKBest object with the chi2 score function to select the top 1000 features, apply SelectKBest to the combined features and the target variable, and display the shape of the selected features.



In [27]:
from sklearn.feature_selection import SelectKBest, chi2

y = df['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})

selector = SelectKBest(score_func=chi2, k=1000)

selected_features = selector.fit_transform(combined_features, y)

display(selected_features.shape)

(18650, 1000)

## Huấn luyện mô hình đơn giản

### Subtask:
Huấn luyện một mô hình phân loại đơn giản (ví dụ: Logistic Regression) trên các đặc trưng đã chọn và nhãn email.


**Reasoning**:
Import LogisticRegression, create an instance, and fit the model using the selected features and the target variable.



In [28]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(selected_features, y)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

## Phân tích đặc trưng và xác định quy tắc

### Subtask:
Phân tích các trọng số (coefficients) hoặc tầm quan trọng của đặc trưng từ mô hình đã huấn luyện để xác định các đặc trưng nào đóng góp nhiều nhất vào việc phân loại và suy ra các quy tắc nhận dạng email đáng ngờ.


**Reasoning**:
Get the coefficients from the trained Logistic Regression model, get the feature names from the TF-IDF vectorizer and the URL features, create a DataFrame to store features and their coefficients, sort the DataFrame by the absolute values of the coefficients, and display the top and bottom features to identify the most important ones.



In [29]:
import numpy as np
import pandas as pd

coefficients = model.coef_[0]

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
url_feature_names = url_features_df.columns.tolist()
all_feature_names = list(tfidf_feature_names) + url_feature_names

selected_indices = selector.get_support(indices=True)
selected_feature_names = [all_feature_names[i] for i in selected_indices]

feature_importance = pd.DataFrame({'feature': selected_feature_names, 'coefficient': coefficients})

feature_importance['abs_coefficient'] = np.abs(feature_importance['coefficient'])

sorted_feature_importance = feature_importance.sort_values(by='abs_coefficient', ascending=False)

display("Top 10 most important features (positive coefficients):")
display(sorted_feature_importance[sorted_feature_importance['coefficient'] > 0].head(10))

display("Top 10 most important features (negative coefficients):")
display(sorted_feature_importance[sorted_feature_importance['coefficient'] < 0].head(10))

'Top 10 most important features (positive coefficients):'

,feature,coefficient,abs_coefficient
989,your,9.194365,9.194365
623,our,5.713258,5.713258
201,company,5.686334,5.686334
394,here,5.611319,5.611319
988,you,5.343440,5.343440
838,statements,5.120842,5.120842
191,click,5.094713,5.094713
569,money,5.008314,5.008314
352,free,4.892743,4.892743
998,has_ip_address_in_url,4.476312,4.476312


'Top 10 most important features (negative coefficients):'

,feature,coefficient,abs_coefficient
305,enron,-11.976672,11.976672
285,ect,-7.228782,7.228782
601,on,-5.311412,5.311412
879,thanks,-5.019949,5.019949
939,vince,-4.925430,4.925430
28,2001,-4.749605,4.749605
475,language,-4.656219,4.656219
916,university,-4.522769,4.522769
881,the,-4.367349,4.367349
27,2000,-4.298495,4.298495


## Summary:

### Data Analysis Key Findings

*   The dataset contains 18,650 emails, each with text content and a classification as either "Safe Email" or "Phishing Email".
*   A small number of emails (16) have missing text content.
*   TF-IDF features were successfully extracted from the email text, resulting in a sparse matrix representation.
*   Several URL-specific features were extracted for each email, including the number of URLs, average URL length, average dots in URLs, average special characters in URLs, presence of an IP address in a URL, and presence of a URL shortening service.
*   The TF-IDF features and URL features were combined into a single feature set.
*   Feature selection using the chi-squared test identified the top 1000 features most relevant for distinguishing between safe and phishing emails.
*   A Logistic Regression model was trained on the selected features.
*   Analysis of the model's coefficients revealed key features associated with phishing emails (positive coefficients) and safe emails (negative coefficients).
    *   Features strongly associated with phishing emails include terms like 'your', 'our', 'company', 'here', 'you', 'statements', 'click', 'money', 'free', and the URL feature 'has\_ip\_address\_in\_url'.
    *   Features strongly associated with safe emails include terms like 'enron', 'ect', 'on', 'thanks', 'vince', '2001', 'language', 'university', 'the', and '2000'.

### Insights or Next Steps

*   Based on the feature importance, simple rules for identifying suspicious emails could be formulated. For example, an email could be flagged as suspicious if it contains terms like "click", "money", or "free", or if any URL within the email contains an IP address or a known shortening service.
*   Further model evaluation using metrics like precision, recall, and F1-score on a separate test set would be beneficial to assess the performance of the trained model and the effectiveness of the selected features and derived rules.


# Task
Đọc dữ liệu từ tệp CSV "/content/Phishing_Email.csv", phân loại email thành 'Phishing Email' và 'Safe Email', đếm số lượng email trong mỗi loại, lưu 'Phishing Email' vào tệp "blackemail.csv" và 'Safe Email' vào tệp "whiteemail.csv", sau đó tóm tắt số lượng email trong mỗi tệp đã lưu.

## Tải dữ liệu

### Subtask:
Tải dữ liệu từ tệp CSV được cung cấp vào DataFrame.


**Reasoning**:
The first step is to load the data from the CSV file into a pandas DataFrame.



In [30]:
df = pd.read_csv("/content/Phishing_Email.csv")

## Phân loại email

### Subtask:
Lọc DataFrame để tách email theo loại ('Phishing Email' và 'Safe Email').


**Reasoning**:
Filter the DataFrame to separate phishing and safe emails based on the 'Email Type' column.



In [31]:
phishing_emails_df = df[df['Email Type'] == 'Phishing Email']
safe_emails_df = df[df['Email Type'] == 'Safe Email']

display(phishing_emails_df.head())
display(safe_emails_df.head())

,Unnamed: 0,Email Text,Email Type
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email
7,7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email
8,8,"we owe you lots of money dear applicant , afte...",Phishing Email
10,10,make her beg you to give it to her everynight ...,Phishing Email


,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
5,5,global risk management operations sally congra...,Safe Email
6,6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email


## Đếm số lượng

### Subtask:
Đếm số lượng email trong mỗi loại ('Phishing Email' và 'Safe Email').


**Reasoning**:
Count the number of rows in the phishing_emails_df and safe_emails_df DataFrames and display the counts.



In [32]:
phishing_email_count = phishing_emails_df.shape[0]
safe_email_count = safe_emails_df.shape[0]

display(f"Number of Phishing Emails: {phishing_email_count}")
display(f"Number of Safe Emails: {safe_email_count}")

'Number of Phishing Emails: 7328'

'Number of Safe Emails: 11322'

## Lưu dữ liệu

### Subtask:
Lưu các email 'Phishing Email' vào tệp "blackemail.csv" và 'Safe Email' vào tệp "whiteemail.csv".


**Reasoning**:
Save the 'Phishing Email' DataFrame to 'blackemail.csv' and the 'Safe Email' DataFrame to 'whiteemail.csv', excluding the index.



In [33]:
phishing_emails_df.to_csv("blackemail.csv", index=False)
safe_emails_df.to_csv("whiteemail.csv", index=False)

## Summary:

### Data Analysis Key Findings

*   The dataset contains 7328 'Phishing Email' and 11322 'Safe Email'.
*   The 'Phishing Email' have been successfully saved to "blackemail.csv".
*   The 'Safe Email' have been successfully saved to "whiteemail.csv".

### Insights or Next Steps

*   The saved CSV files can be used for further analysis or building machine learning models to classify emails.
*   Investigate the characteristics of phishing emails in "blackemail.csv" to identify common patterns or keywords.
